In [16]:
import json
import networkx as nx
from pyvis.network import Network
import tempfile
import webbrowser

# Load JSON data from a file
json_file_path = "sampleevent.json"  # Change this to the correct file path

with open(json_file_path, "r") as file:
    json_data = json.load(file)

rules_data = json_data["rules"]
data = json_data["data"]
original_data = json_data["data"]  # Store original data state for debugging

# Create a Directed Graph
G = nx.DiGraph()
rule_execution_status = {}  # Track rule pass/fail status
attribute_to_rule = {}  # Map attributes to the rule that modifies them
rule_data_changes = {}  # Track data changes globally
rule_initial_data = {}  # Track initial data passed to each rule

# Evaluate rules and populate graph
for rule in rules_data:
    rule_id = rule["rule_id"]
    G.add_node(rule_id, label=rule_id)
    rule_pass = True
    rule_initial_data[rule_id] = {}
    
    # Evaluate conditions
    for condition in rule["conditions"]:
        class_name = condition["class"]
        attribute = condition["attribute"]
        operator = condition["operator"]
        value = condition["value"]
        data_value = data.get(class_name, {}).get(attribute, None)
        
        # Store initial data passed to rule
        rule_initial_data[rule_id][attribute] = data_value
        
        if operator == "equals":
            rule_pass &= (data_value == value)
        elif operator == "greater_than":
            rule_pass &= (data_value > value)
        elif operator == "less_than":
            rule_pass &= (data_value < value)
        elif operator == "between":
            rule_pass &= (value[0] <= data_value <= value[1])
        elif operator == "in":
            rule_pass &= (data_value in value)
        elif operator == "not_in":
            rule_pass &= (data_value not in value)
        else:
            rule_pass = False
    
    rule_execution_status[rule_id] = rule_pass

    # Execute actions if rule passes
    if rule_pass:
        for action in rule["actions"]:
            if action["type"] == "operation":
                class_name = action["class"]
                attribute = action["attribute"]
                operation = action["operation"]
                value = action["value"]
                previous_value = data[class_name][attribute]
                
                if operation == "constant":
                    data[class_name][attribute] = value
                elif operation == "multiply":
                    data[class_name][attribute] *= value
                
                rule_data_changes[attribute] = f"{previous_value} → {data[class_name][attribute]}"
                attribute_to_rule[attribute] = rule_id

# Create interactive Pyvis graph
net = Network(notebook=True, directed=True, height='600px', width='100%', bgcolor='#222222', font_color='white')
net.toggle_physics(False)

for node in G.nodes:
    color = "green" if rule_execution_status[node] else "red"
    rule_details = next((r for r in rules_data if r["rule_id"] == node), None)
    rule_info = json.dumps(rule_details, indent=2) if rule_details else "Rule details not found"
    data_passed = json.dumps(rule_initial_data.get(node, {}), indent=2)
    # data_changes = rule_data_changes.get(node, "No data changes")
    node_title = f"{node}\n{rule_info}\nInput Data: {data_passed}"
    # node_title = f"{node}\n{rule_info}\nInput Data: {data_passed}\n{data_changes}"
    net.add_node(node, label=node, title=node_title, color=color)

# Add edges with labels
for attribute, rule_id in attribute_to_rule.items():
    for rule in rules_data:
        if any(cond["attribute"] == attribute for cond in rule["conditions"]):
            net.add_edge(rule_id, rule["rule_id"], title=f"Depends on {attribute}")

# Save and open interactive visualization
with tempfile.NamedTemporaryFile(delete=False, suffix=".html") as temp_file:
    net.show(temp_file.name)
    webbrowser.open(temp_file.name)


KeyError: 'value'

In [25]:
import json
import networkx as nx
from pyvis.network import Network
import tempfile
import webbrowser

# Load JSON data from a file
json_file_path = "samplerules.json"  # Change this to the correct file path

with open(json_file_path, "r") as file:
    json_data = json.load(file)

rules_data = json_data["rules"]
data = json_data["data"]
original_data = json_data["data"]  # Store original data state for debugging

# Create a Directed Graph
G = nx.DiGraph()
rule_execution_status = {}  # Track rule pass/fail status
attribute_to_rule = {}  # Map attributes to the rule that modifies them
rule_data_changes = {}  # Track data changes globally
rule_initial_data = {}  # Track initial data passed to each rule

# Evaluate rules and populate graph
for rule in rules_data:
    rule_id = rule["rule_id"]
    G.add_node(rule_id, label=rule_id)
    rule_pass = True
    rule_initial_data[rule_id] = {}
    
    # Evaluate conditions
    for condition in rule["conditions"]:
        class_name = condition["class"]
        attribute = condition["attribute"]
        operator = condition["operator"]
        value = condition.get("value", None)  # Prevent KeyError
        rule_compare = condition.get("rule_compare", "simple")
        compare_class = condition.get("compare_class", None)
        compare_attribute = condition.get("compare_attribute", None)
        
        data_value = data.get(class_name, {}).get(attribute, None)
        compare_value = data.get(compare_class, {}).get(compare_attribute, None) if compare_class and compare_attribute else None
        
        # Store initial data passed to rule
        rule_initial_data[rule_id][attribute] = data_value
        
        if rule_compare == "simple":
            if operator == "equals":
                rule_pass &= (data_value == value)
            elif operator == "greater_than":
                rule_pass &= (data_value > value)
            elif operator == "less_than":
                rule_pass &= (data_value < value)
            elif operator == "between" and isinstance(value, list) and len(value) == 2:
                rule_pass &= (value[0] <= data_value <= value[1])
            elif operator == "in" and isinstance(value, list):
                rule_pass &= (data_value in value)
            elif operator == "not_in" and isinstance(value, list):
                rule_pass &= (data_value not in value)
            else:
                rule_pass = False
        elif rule_compare == "complex" and compare_value is not None:
            if operator == "equals":
                rule_pass &= (data_value == compare_value)
            elif operator == "greater_than":
                rule_pass &= (data_value > compare_value)
            elif operator == "less_than":
                rule_pass &= (data_value < compare_value)
            else:
                rule_pass = False
    
    rule_execution_status[rule_id] = rule_pass

    # Execute actions if rule passes
    if rule_pass:
        for action in rule["actions"]:
            if action["type"] == "operation":
                class_name = action.get("class", None)
                attribute = action["attribute"]
                operation = action["operation"]
                value = action["value"]
                previous_value = data.get(class_name, {}).get(attribute, None)
                
                if class_name and attribute in data.get(class_name, {}):
                    if operation == "constant":
                        data[class_name][attribute] = value
                    elif operation == "multiply":
                        data[class_name][attribute] *= value
                    elif operation == "add":
                        data[class_name][attribute] += value
                    
                    rule_data_changes[attribute] = f"{previous_value} → {data[class_name][attribute]}"
                    attribute_to_rule[attribute] = rule_id

# Create interactive Pyvis graph
net = Network(notebook=True, directed=True, height='600px', width='100%', bgcolor='#222222', font_color='white')
net.toggle_physics(False)

for node in G.nodes:
    color = "green" if rule_execution_status[node] else "red"
    rule_details = next((r for r in rules_data if r["rule_id"] == node), None)
    rule_info = json.dumps(rule_details, indent=2) if rule_details else "Rule details not found"
    data_passed = json.dumps(rule_initial_data.get(node, {}), indent=2)
    data_changes = rule_data_changes.get(node, "No data changes")
    node_title = f"{node}\n{rule_info}\nInput Data: {data_passed}\n{data_changes}"
    net.add_node(node, label=node, title=node_title, color=color)

# Add edges with labels
for attribute, rule_id in attribute_to_rule.items():
    for rule in rules_data:
        if any(cond["attribute"] == attribute for cond in rule["conditions"]):
            net.add_edge(rule_id, rule["rule_id"], title=f"Depends on {attribute}")

# Save and open interactive visualization
with tempfile.NamedTemporaryFile(delete=False, suffix=".html") as temp_file:
    net.show(temp_file.name)
    webbrowser.open(temp_file.name)


C:\Users\rajes\AppData\Local\Temp\tmp608kuukf.html


In [ ]:
# import json
# import networkx as nx
# from pyvis.network import Network
# import tempfile
# import webbrowser

# # Load JSON data from a file
# json_file_path = "sampleevent.json"  # Change this to the correct file path

# with open(json_file_path, "r") as file:
#     json_data = json.load(file)

# rules_data = json_data["rules"]
# data = json_data["data"]
# original_data = json_data["data"]  # Store original data state for debugging

# # Create a Directed Graph
# G = nx.DiGraph()
# rule_execution_status = {}  # Track rule pass/fail status
# attribute_to_rule = {}  # Map attributes to the rule that modifies them
# rule_data_changes = {}  # Track data changes globally
# rule_initial_data = {}  # Track initial data passed to each rule
# rule_guidance = {}  # Store guidance text per rule
# rule_changeable = {}  # Track changeable status of rules

# # Evaluate rules and populate graph
# for rule in rules_data:
#     rule_id = rule["rule_id"]
#     G.add_node(rule_id, label=rule_id)
#     rule_pass = True
#     rule_initial_data[rule_id] = {}
#     rule_guidance[rule_id] = rule.get("guidance", "No guidance available.")
#     rule_changeable[rule_id] = rule.get("changeable", True)  # Default to changeable
    
#     # Evaluate conditions
#     for condition in rule["conditions"]:
#         class_name = condition["class"]
#         attribute = condition["attribute"]
#         operator = condition["operator"]
#         value = condition.get("value", None)  # Prevent KeyError
#         rule_compare = condition.get("rule_compare", "simple")
#         compare_class = condition.get("compare_class", None)
#         compare_attribute = condition.get("compare_attribute", None)
        
#         data_value = data.get(class_name, {}).get(attribute, None)
#         compare_value = data.get(compare_class, {}).get(compare_attribute, None) if compare_class and compare_attribute else None
        
#         # Store initial data passed to rule
#         rule_initial_data[rule_id][attribute] = data_value
        
#         if rule_compare == "simple":
#             if operator == "equals":
#                 rule_pass &= (data_value == value)
#             elif operator == "greater_than":
#                 rule_pass &= (data_value > value)
#             elif operator == "less_than":
#                 rule_pass &= (data_value < value)
#             elif operator == "between" and isinstance(value, list) and len(value) == 2:
#                 rule_pass &= (value[0] <= data_value <= value[1])
#             elif operator == "in" and isinstance(value, list):
#                 rule_pass &= (data_value in value)
#             elif operator == "not_in" and isinstance(value, list):
#                 rule_pass &= (data_value not in value)
#             else:
#                 rule_pass = False
#         elif rule_compare == "complex" and compare_value is not None:
#             if operator == "equals":
#                 rule_pass &= (data_value == compare_value)
#             elif operator == "greater_than":
#                 rule_pass &= (data_value > compare_value)
#             elif operator == "less_than":
#                 rule_pass &= (data_value < compare_value)
#             else:
#                 rule_pass = False
    
#     rule_execution_status[rule_id] = rule_pass

#     # Execute actions if rule passes
#     if rule_pass:
#         for action in rule["actions"]:
#             if action["type"] == "operation":
#                 class_name = action.get("class", None)
#                 attribute = action["attribute"]
#                 operation = action["operation"]
#                 value = action["value"]
#                 previous_value = data.get(class_name, {}).get(attribute, None)
                
#                 if class_name and attribute in data.get(class_name, {}):
#                     if operation == "constant":
#                         data[class_name][attribute] = value
#                     elif operation == "multiply":
#                         data[class_name][attribute] *= value
#                     elif operation == "add":
#                         data[class_name][attribute] += value
                    
#                     rule_data_changes[attribute] = f"{previous_value} → {data[class_name][attribute]}"
#                     attribute_to_rule[attribute] = rule_id

# # Prune guidance based on non-changeable rules
# final_guidance = []
# processed_rules = set()

# for rule_id, passed in rule_execution_status.items():
#     if not passed:  # Rule failed
#         parent_rule = next((parent for parent, child in attribute_to_rule.items() if child == rule_id), None)
        
#         if parent_rule and not rule_changeable.get(parent_rule, True):
#             continue  # Skip guidance if parent is non-changeable
        
#         if rule_id not in processed_rules:
#             final_guidance.append(rule_guidance[rule_id])
#             processed_rules.add(rule_id)

# # Create interactive Pyvis graph
# net = Network(notebook=True, directed=True, height='600px', width='100%', bgcolor='#222222', font_color='white')
# net.toggle_physics(False)

# for node in G.nodes:
#     color = "green" if rule_execution_status[node] else "red"
#     if not rule_changeable[node]:
#         color = "blue"  # Highlight non-changeable rules
    
#     rule_details = next((r for r in rules_data if r["rule_id"] == node), None)
#     rule_info = json.dumps(rule_details, indent=2) if rule_details else "Rule details not found"
#     data_passed = json.dumps(rule_initial_data.get(node, {}), indent=2)
#     data_changes = rule_data_changes.get(node, "No data changes")
#     node_title = f"{node}\n{rule_info}\nInput Data: {data_passed}\n{data_changes}\nGuidance: {rule_guidance.get(node, 'No guidance')}"
#     net.add_node(node, label=node, title=node_title, color=color)

# # Add edges with labels
# for attribute, rule_id in attribute_to_rule.items():
#     for rule in rules_data:
#         if any(cond["attribute"] == attribute for cond in rule["conditions"]):
#             net.add_edge(rule_id, rule["rule_id"], title=f"Depends on {attribute}")

# # Save and open interactive visualization
# with tempfile.NamedTemporaryFile(delete=False, suffix=".html") as temp_file:
#     net.show(temp_file.name)
#     webbrowser.open(temp_file.name)

# # Print final guidance for failed rules
# print("Final Guidance Messages:")
# for guidance in final_guidance:
#     print(f"- {guidance}")


C:\Users\rajes\AppData\Local\Temp\tmpmzcxpro6.html
Final Guidance Messages:
- Check product ID.


In [25]:
import json
import networkx as nx
from pyvis.network import Network
import tempfile
import webbrowser

# Load JSON data from a file
json_file_path = "sampleevent3.json"  # Change this to the correct file path

with open(json_file_path, "r") as file:
    json_data = json.load(file)

rules_data = json_data["rules"]
data = json_data["data"]
original_data = json_data["data"]  # Store original data state for debugging

# Create a Directed Graph
G = nx.DiGraph()
rule_execution_status = {}  # Track rule pass/fail status
attribute_to_rule = {}  # Map attributes to the rule that modifies them
rule_data_changes = {}  # Track data changes globally
rule_initial_data = {}  # Track initial data passed to each rule
rule_guidance = {}  # Store guidance text per rule
rule_changeable = {}  # Track changeable status of rules

# Evaluate rules and populate graph
for rule in rules_data:
    rule_id = rule["rule_id"]
    G.add_node(rule_id, label=rule_id)
    rule_pass = True
    rule_initial_data[rule_id] = {}
    rule_guidance[rule_id] = rule.get("guidance", "No guidance available.")
    rule_changeable[rule_id] = rule.get("changeable", True)  # Default to changeable
    
    # Evaluate conditions
    for condition in rule["conditions"]:
        class_name = condition["class"]
        attribute = condition["attribute"]
        operator = condition["operator"]
        value = condition.get("value", None)  # Prevent KeyError
        rule_compare = condition.get("rule_compare", "simple")
        compare_class = condition.get("compare_class", None)
        compare_attribute = condition.get("compare_attribute", None)
        
        data_value = data.get(class_name, {}).get(attribute, None)
        compare_value = data.get(compare_class, {}).get(compare_attribute, None) if compare_class and compare_attribute else None
        
        # Store initial data passed to rule
        rule_initial_data[rule_id][attribute] = str(data_value)  # Ensure it's a string for display
        
        if rule_compare == "simple":
            if operator == "equals":
                rule_pass &= (data_value == value)
            elif operator == "greater_than":
                rule_pass &= (data_value > value)
            elif operator == "less_than":
                rule_pass &= (data_value < value)
            elif operator == "between" and isinstance(value, list) and len(value) == 2:
                rule_pass &= (value[0] <= data_value <= value[1])
            elif operator == "in" and isinstance(value, list):
                rule_pass &= (data_value in value)
            elif operator == "not_in" and isinstance(value, list):
                rule_pass &= (data_value not in value)
            else:
                rule_pass = False
        elif rule_compare == "complex" and compare_value is not None:
            if operator == "equals":
                rule_pass &= (data_value == compare_value)
            elif operator == "greater_than":
                rule_pass &= (data_value > compare_value)
            elif operator == "less_than":
                rule_pass &= (data_value < compare_value)
            else:
                rule_pass = False
    
    rule_execution_status[rule_id] = rule_pass

    # Execute actions if rule passes
    if rule_pass:
        for action in rule["actions"]:
            if action["type"] == "operation":
                class_name = action.get("class", None)
                attribute = action["attribute"]
                operation = action["operation"]
                value = action["value"]
                previous_value = data.get(class_name, {}).get(attribute, None)
                
                if class_name and attribute in data.get(class_name, {}):
                    if operation == "constant":
                        data[class_name][attribute] = value
                    elif operation == "multiply":
                        data[class_name][attribute] *= value
                    elif operation == "add":
                        data[class_name][attribute] += value
                    
                    rule_data_changes[attribute] = f"{previous_value} → {data[class_name][attribute]}"
                    attribute_to_rule[attribute] = rule_id

# Prune guidance based on non-changeable rules
final_guidance = []
processed_rules = set()

for rule_id, passed in rule_execution_status.items():
    if not passed:  # Rule failed
        parent_rule = next((parent for parent, child in attribute_to_rule.items() if child == rule_id), None)
        
        if parent_rule and not rule_changeable.get(parent_rule, True):
            continue  # Skip guidance if parent is non-changeable
        
        if rule_id not in processed_rules:
            final_guidance.append(rule_guidance[rule_id])
            processed_rules.add(rule_id)

# Create interactive Pyvis graph
net = Network(notebook=True, directed=True, height='600px', width='100%', bgcolor='#222222', font_color='white')
net.toggle_physics(False)

for node in G.nodes:
    color = "green" if rule_execution_status[node] else "red"
    if not rule_changeable[node]:
        color = "blue"  # Highlight non-changeable rules
    
    rule_details = next((r for r in rules_data if r["rule_id"] == node), None)
    rule_info = json.dumps(rule_details, indent=2) if rule_details else "Rule details not found"
    data_passed = "\n".join([f"{k}: {v}" for k, v in rule_initial_data.get(node, {}).items()])  # Proper string format
    data_changes = rule_data_changes.get(node, "No data changes")
    node_title = f"{node}\n\nRule: {rule_info}\n\nInput Data:\n{data_passed}\n\nData Changes: {data_changes}\n\nGuidance: {rule_guidance.get(node, 'No guidance')}"
    net.add_node(node, label=node, title=node_title, color=color)

# Add edges with labels
for attribute, rule_id in attribute_to_rule.items():
    for rule in rules_data:
        if any(cond["attribute"] == attribute for cond in rule["conditions"]):
            net.add_edge(rule_id, rule["rule_id"], title=f"Depends on {attribute}")

# Save and open interactive visualization
with tempfile.NamedTemporaryFile(delete=False, suffix=".html") as temp_file:
    net.show(temp_file.name)
    webbrowser.open(temp_file.name)

# Print final guidance for failed rules
print("Final Guidance Messages:")
for guidance in final_guidance:
    print(f"- {guidance}")


C:\Users\rajes\AppData\Local\Temp\tmp0aphux16.html
Final Guidance Messages:
- Customers with VIP status receive an extra discount.
- Orders with discounts above $50 get a special discount label.
- Orders above $600 qualify for free shipping.
